In [ ]:
# Import packages
from __future__ import annotations
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from causalml.inference.tree import UpliftRandomForestClassifier
from pathlib import Path
import sys
import pandas as pd
import warnings
import os 
import plotly.io as pio

pd.options.mode.chained_assignment = None
warnings.simplefilter("ignore")

In [ ]:
np.random.seed(42)

In [ ]:
# Get custom functions
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Thesis code" and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from Functions.data_utils import (
    plot_incremental_response_rate,
    uplift_by_decile_bin,
    coerce_metrics_to_numeric,
)

In [ ]:
file_path = r"Data/covariates_modeling_uplift_models_2026-03-13.csv"
df = pd.read_csv(file_path)

In [ ]:
df['reactivated'].value_counts(normalize = True)

In [ ]:
# convert for easy processing 
treatment_converter = {
    "BNLX_ChurnP_10_test_export.csv": "treatment_1",
    "BNLX_ChurnP_10_controle_export.csv": "control_1", 
    
    "BNLX_ChurnP_25_test_export.csv": "treatment_2",
    "BNLX_ChurnP_25_controle_export.csv": "control_2",

    "BNLX_ChurnP_5eu_test_export.csv": "treatment_3",
    "BNLX_ChurnP_5eu_controle_export.csv": "control_3",

    "BNLX_ChurnP_10eu_test_export.csv": "treatment_4",
    "BNLX_ChurnP_10eu_controle_export.csv": "control_4",
    
    "BNLX_ChurnP_250_test_export.csv": "treatment_5",
    "BNLX_ChurnP_250_controle_export.csv": "control_5",

    "BNLX_ChurnP_500_test_export.csv": "treatment_6",
    "BNLX_ChurnP_500_controle_export.csv": "control_6",    

    "BNLX_ChurnP_SKUe_test_export.csv": "treatment_7",
    "BNLX_ChurnP_SKUe_controle_export.csv": "control_7",    

    "BNLX_ChurnP_SKUd_test_export.csv": "treatment_8",
    "BNLX_ChurnP_SKUd_controle_export.csv": "control_8",
    
    "BNLX_ChurnP_niks_test_export.csv": "treatment_9",
    "BNLX_ChurnP_niks_controle_export.csv": "control_9",
}

df["treatment"] = df["treatment_indicator"].map(treatment_converter)

In [ ]:
# Variables for modelling
categorical_cols  = ['has_rfl','gender','country_sk']
numeric_cols = [ 'recency', 'frequency', 'monetary_value',  'total_volume', 'length_of_relationship', 'online_sales', 'retail_sales', 
      'food_total', 'vhms_total', 'sports_total', 'beauty_total', 'frequency_52wk', 'monetary_value_52wk', 'volume_52wk', 'online_sales_52w', 
       'retail_sales_52w', 'frequency_53w_104w', 'monetary_value_53w_104w', 'volume_53w_104w', 'online_sales_53w_104w','retail_sales_53w_104w']

In [ ]:
# Fix numeric output
df = coerce_metrics_to_numeric(df, numeric_cols)
df[categorical_cols] = df[categorical_cols].astype("object")
df[numeric_cols] = df[numeric_cols].astype("int64")

In [ ]:
(100 * df.groupby('treatment_indicator')['reactivated'].mean()).round(2)

In [ ]:
# Split for modeling
X = df[numeric_cols + categorical_cols]
y = df['reactivated']
t = df['treatment']

In [ ]:
def five_fold_RF_uplift(
    # Five-fold CV the Uplift random forest over the treatment and control dataset.
    df: pd.DataFrame,
    X: pd.DataFrame,
    y,
    t,
    *,
    n_splits: int = 5,
    random_state: int = 42,
    control_name: str = "control_0",
) -> pd.DataFrame:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    y_s = pd.Series(y, index=X.index)
    t_s = pd.Series(t, index=X.index)
    # Create joint stratification label: treatment vs control AND outcome
    strata = (t_s.astype(str) + "_" + y_s.astype(int).astype(str))

    fold_results: list[pd.DataFrame] = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, strata), start=1):
        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()

        y_train = y_s.iloc[train_idx]
        t_train = t_s.iloc[train_idx]

        categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns
        numeric_cols = X_train.columns.difference(categorical_cols)

        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

        def _prep_cat(X_part: pd.DataFrame) -> pd.DataFrame:
            if len(categorical_cols) == 0:
                return X_part
            return X_part.assign(
                **{c: X_part[c].astype("string").fillna("MISSING") for c in categorical_cols}
            )

        def encode_X(X_part: pd.DataFrame) -> np.ndarray:
            X_part = _prep_cat(X_part)
            X_num = X_part[numeric_cols].to_numpy()

            if len(categorical_cols) == 0:
                return X_num

            X_cat = ohe.transform(X_part[categorical_cols])
            return np.hstack([X_num, X_cat])


        X_train_prep = _prep_cat(X_train)
        ohe.fit(X_train_prep[categorical_cols])

        X_train_enc = encode_X(X_train)
        X_test_enc = encode_X(X_test)

        uplift_model = UpliftRandomForestClassifier(control_name=control_name, n_estimators= 100, random_state=42, n_jobs=1)
        uplift_model.fit(
            X_train_enc,
            treatment=t_train.astype(str).values,
            y=y_train.astype(int).values,
        )

        uplift_pred = np.asarray(uplift_model.predict(X_test_enc))
        cate = uplift_pred.ravel()

        test_df = df.loc[X_test.index].copy()
        test_df["cate"] = cate
        test_df["fold"] = fold
        fold_results.append(test_df)

    return (
        pd.concat(fold_results, axis=0)
          .sort_values("cate", ascending=False)
    )




In [ ]:
# loop over each combination of treatment and control at the last number and appply the five fold CV RF uplift model
cv_test_df = pd.concat(
    [
        (
            print("Incentive", k),
            five_fold_RF_uplift(
                df=df.loc[t.isin([f"treatment_{k}", f"control_{k}"])],
                X=X.loc[t.isin([f"treatment_{k}", f"control_{k}"])],
                y=y.loc[t.isin([f"treatment_{k}", f"control_{k}"])],
                t=t.loc[t.isin([f"treatment_{k}", f"control_{k}"])],
                control_name=f"control_{k}",
            ).assign(experiment_k=k)
        )[1]
        for k in range(1, 10)
        if t.isin([f"treatment_{k}", f"control_{k}"]).any()
    ],
    axis=0,
).sort_values(["experiment_k", "cate"], ascending=[True, False])

In [ ]:
# Convert back treatment names
INCENTIVE_NAMES = {
    1: "10%_discount",
    2: "25%_discount",
    3: "5eu_voucher",
    4: "10eu_voucher",
    5: "250_loyalty_pts",
    6: "500_loyalty_pts",
    7: "Vitamin E",
    8: "Vitamine D",
    9: "Renewal communication"
}

# generate the qini bins needed for qini plots
qini_bins = (
    cv_test_df
    .groupby("experiment_k", group_keys=False)
    .apply(lambda d: uplift_by_decile_bin(d, size=10).assign(incentive_k=d.name))
    .reset_index(drop=True)
)

# Replace numeric codes with original names
qini_bins["incentive_k"] = qini_bins["incentive_k"].map(INCENTIVE_NAMES)

In [ ]:
qini_bins.to_excel(
    os.path.join("Output/qini_bins_binary_uplift.xlsx"),
    index=False,
)

In [ ]:
qini_bins[qini_bins['bin']==3]

In [ ]:
# make qini plots per incentive
pio.renderers.default = "jupyterlab"
out_dir = Path("Output/Output_qini_curves")
out_dir.mkdir(parents=True, exist_ok=True)

for k, qini_k in qini_bins.groupby("incentive_k", sort=False):
    fig = plot_incremental_response_rate(qini_k)

    incentive = f"incentive_{k}"
    fig.update_layout(
        title=f"Qini curve ({incentive})"
    )

    fig.write_html(out_dir / f"{incentive}_incremental_response_rate.html")

    fig.show()